# 🔧 Bosch Accelerometer — Ollama Local LLM Query Engine

**This notebook is a drop-in upgrade of the original Bosch Accelerometer Motion Event Explainer.**

**What changed:** Section 8 (LlamaIndex integration) is replaced with a fully local LLM stack:
- **Ollama** serves `llama3.2:1b` on Colab's free T4 GPU
- **HuggingFace `bge-small-en-v1.5`** provides free local embeddings
- **No OpenAI API key required**

**Pipeline:**
`Synthetic data → Normalize → Features → Rule-based detection → Summaries → LlamaIndex (Ollama) → Query`

> **Important:** Before running, go to **Runtime > Change runtime type** and set Hardware accelerator to **GPU (T4)**. Ollama runs on CPU too but will be slower.

---
## 1. Install Ollama and Pull the Model

This cell installs Ollama, starts its background server, and downloads `llama3.2:1b` (~800 MB).

- `llama3.2:1b` is the smallest Llama 3.2 model — fast on T4, good answer quality for summarization tasks.
- The `nohup ... &` pattern keeps Ollama running while the rest of the notebook executes.
- `time.sleep(6)` gives the server time to initialize before we make requests.

In [ ]:
# ── Step 1: Install Ollama ────────────────────────────────────────────────────
print('Installing Ollama...')
!curl -fsSL https://ollama.com/install.sh | sh

# ── Step 2: Start the Ollama server in the background ────────────────────────
print('Starting Ollama server...')
!nohup ollama serve > /tmp/ollama.log 2>&1 &

import time
time.sleep(6)   # wait for server to be ready

# ── Step 3: Pull the model (downloads ~800 MB on first run) ──────────────────
# Swap 'llama3.2:1b' for 'llama3.2:3b' if you want higher quality answers
# and have patience — the 3b model is ~2 GB.
MODEL_NAME = 'llama3.2:1b'
print(f'Pulling {MODEL_NAME} — this may take a few minutes on first run...')
!ollama pull {MODEL_NAME}

print(f'\n✅ Ollama is running and {MODEL_NAME} is ready.')

---
## 2. Install Python Packages

Installs:
- `llama-index-core` — base LlamaIndex framework
- `llama-index-llms-ollama` — Ollama LLM connector
- `llama-index-embeddings-huggingface` — local HuggingFace embeddings (no API key)
- `sentence-transformers` — required by the HuggingFace embedding model
- Standard data stack: `pandas`, `numpy`, `matplotlib`

In [ ]:
!pip install \
    pandas numpy matplotlib \
    llama-index-core \
    llama-index-llms-ollama \
    llama-index-embeddings-huggingface \
    sentence-transformers \
    --quiet

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('✅ All packages installed and imported.')

---
## 3. Synthetic Data Generation

Generates four motion segments that mimic real Bosch accelerometer behavior.
All produce a DataFrame with columns: `timestamp, accel_x, accel_y, accel_z`.

| Segment | Behavior |
|---|---|
| `idle` | Device at rest, near-zero XY, gravity on Z |
| `walking_like` | Rhythmic ~2 Hz oscillation |
| `impact` | Short high-magnitude spike with damped ringing |
| `shaking` | High-frequency high-amplitude uniform noise |

In [ ]:
# ── Tunable parameters ────────────────────────────────────────────────────────
SAMPLE_RATE_HZ = 100
GRAVITY = 9.81
REQUIRED_COLS = {'timestamp', 'accel_x', 'accel_y', 'accel_z'}
ACCEL_COLS = ['accel_x', 'accel_y', 'accel_z']

SEG_DURATIONS = {
    'idle':         5.0,
    'walking_like': 8.0,
    'impact':       2.0,
    'shaking':      4.0,
}


def make_idle(n):
    noise = 0.05
    return np.column_stack([
        np.random.normal(0, noise, n),
        np.random.normal(0, noise, n),
        np.random.normal(GRAVITY, noise, n)
    ])


def make_walking_like(n):
    t = np.linspace(0, n / SAMPLE_RATE_HZ, n)
    freq, amp, noise = 2.0, 1.5, 0.2
    return np.column_stack([
        amp * np.sin(2 * np.pi * freq * t) + np.random.normal(0, noise, n),
        amp * np.cos(2 * np.pi * freq * t) + np.random.normal(0, noise, n),
        GRAVITY + 0.5 * np.sin(2 * np.pi * freq * t) + np.random.normal(0, noise, n)
    ])


def make_impact(n):
    x = np.random.normal(0, 0.1, n)
    y = np.random.normal(0, 0.1, n)
    z = np.random.normal(GRAVITY, 0.1, n)
    spike_idx   = max(1, int(0.20 * n))
    spike_width = max(2, int(0.03 * n))
    envelope = np.zeros(n)
    for i in range(spike_width):
        idx = spike_idx + i
        if idx < n:
            envelope[idx] = 30.0 * np.exp(-4.0 * i / spike_width)
    z += envelope
    x += envelope * 0.4
    return np.column_stack([x, y, z])


def make_shaking(n):
    amp = 5.0
    return np.column_stack([
        np.random.uniform(-amp, amp, n),
        np.random.uniform(-amp, amp, n),
        GRAVITY + np.random.uniform(-amp, amp, n)
    ])


def generate_synthetic_data(sample_rate=SAMPLE_RATE_HZ, durations=SEG_DURATIONS, seed=RANDOM_SEED):
    np.random.seed(seed)
    generators = {
        'idle': make_idle, 'walking_like': make_walking_like,
        'impact': make_impact, 'shaking': make_shaking
    }
    segments, t_offset = [], 0.0
    for name, dur in durations.items():
        n  = int(dur * sample_rate)
        ts = t_offset + np.arange(n) / sample_rate
        df_seg = pd.DataFrame(generators[name](n), columns=ACCEL_COLS)
        df_seg.insert(0, 'timestamp', np.round(ts, 4))
        df_seg['_segment_label'] = name
        segments.append(df_seg)
        t_offset = ts[-1] + 1 / sample_rate
    return pd.concat(segments, ignore_index=True)


raw_df = generate_synthetic_data()
print(f'✅ Generated {len(raw_df):,} samples — duration: {raw_df["timestamp"].max():.2f}s')
raw_df.head()

---
## 4. Normalize + Feature Engineering

- Z-score normalizes `accel_x/y/z` (raw values preserved as `_raw_*`)
- Adds `accel_magnitude = √(x²+y²+z²)`
- Adds `rolling_mean` and `rolling_std` over a 50-sample window

In [ ]:
ROLLING_WINDOW = 50   # ~0.5 s at 100 Hz — lower = more reactive


def normalize_and_engineer(df: pd.DataFrame, window: int = ROLLING_WINDOW) -> pd.DataFrame:
    df = df.copy()
    df.dropna(subset=['timestamp'] + ACCEL_COLS, inplace=True)
    df.sort_values('timestamp', inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Z-score normalization
    for col in ACCEL_COLS:
        df[f'_raw_{col}'] = df[col]
        mu, sigma = df[col].mean(), df[col].std()
        df[col] = (df[col] - mu) / (sigma if sigma > 0 else 1.0)

    # Derived features
    df['accel_magnitude'] = np.sqrt(df['accel_x']**2 + df['accel_y']**2 + df['accel_z']**2)
    df['rolling_mean'] = df['accel_magnitude'].rolling(window, min_periods=1).mean()
    df['rolling_std']  = df['accel_magnitude'].rolling(window, min_periods=1).std().fillna(0)
    return df


feat_df = normalize_and_engineer(raw_df)
print('✅ Normalization and feature engineering complete.')
feat_df[['timestamp', 'accel_magnitude', 'rolling_mean', 'rolling_std']].head(8)

---
## 5. Rule-Based Event Detection

Each sample is classified by comparing `rolling_mean` and `rolling_std` against tunable thresholds.
Consecutive same-label runs are merged into `MotionEvent` objects.

**Threshold tuning guide:**
- Lower `IMPACT_MAG_MIN` → catch weaker impacts
- Lower `SHAKE_STD_MIN` → catch mild vibration
- Raise `MIN_EVENT_SAMPLES` → suppress short noise bursts

In [ ]:
# ── Tunable thresholds ────────────────────────────────────────────────────────
IDLE_STD_MAX      = 0.15
IMPACT_MAG_MIN    = 2.5
IMPACT_STD_MIN    = 1.0
SHAKE_STD_MIN     = 1.8
WALK_MAG_MIN      = 0.8
WALK_MAG_MAX      = 2.5
WALK_STD_MIN      = 0.10
WALK_STD_MAX      = 1.8
MIN_EVENT_SAMPLES = 10


@dataclass
class MotionEvent:
    start_time:       float
    end_time:         float
    event_type:       str
    max_magnitude:    float
    mean_magnitude:   float
    explanation_seed: str


def classify_sample(row) -> str:
    rm, rs = row['rolling_mean'], row['rolling_std']
    if rm > IMPACT_MAG_MIN and rs > IMPACT_STD_MIN:
        return 'impact'
    if rs > SHAKE_STD_MIN:
        return 'shaking'
    if WALK_MAG_MIN <= rm <= WALK_MAG_MAX and WALK_STD_MIN <= rs <= WALK_STD_MAX:
        return 'walking_like'
    if rs < IDLE_STD_MAX:
        return 'idle'
    return 'unknown'


SEED_MAP = {
    'idle':         'the device was stationary with minimal movement',
    'walking_like': 'rhythmic oscillation consistent with walking or repetitive motion',
    'impact':       'a sudden high-magnitude spike indicating a physical impact or drop',
    'shaking':      'high-frequency high-amplitude vibration indicating vigorous shaking',
    'unknown':      'unclassified motion not matching any known pattern',
}


def detect_events(df: pd.DataFrame):
    df = df.copy()
    df['event_label'] = df.apply(classify_sample, axis=1)
    df['_run_id'] = (df['event_label'] != df['event_label'].shift()).cumsum()
    events = []
    for _, grp in df.groupby('_run_id'):
        if len(grp) < MIN_EVENT_SAMPLES:
            continue
        etype = grp['event_label'].iloc[0]
        mag   = grp['accel_magnitude']
        events.append(MotionEvent(
            start_time       = grp['timestamp'].iloc[0],
            end_time         = grp['timestamp'].iloc[-1],
            event_type       = etype,
            max_magnitude    = round(float(mag.max()), 4),
            mean_magnitude   = round(float(mag.mean()), 4),
            explanation_seed = SEED_MAP.get(etype, ''),
        ))
    return events, df


events, labeled_df = detect_events(feat_df)
print(f'✅ Detected {len(events)} events.')
pd.DataFrame([{
    'event_type': e.event_type,
    'start_time': e.start_time,
    'end_time':   e.end_time,
    'duration_s': round(e.end_time - e.start_time, 3),
    'max_mag':    e.max_magnitude,
    'mean_mag':   e.mean_magnitude,
} for e in events])

---
## 6. Event Summarization + Visualization

Each `MotionEvent` is converted to a plain-English sentence. These sentences become the documents indexed by LlamaIndex.
The 3-panel plot shows axes, magnitude with event bands, and rolling statistics.

In [ ]:
def summarize_event(e: MotionEvent) -> str:
    dur = round(e.end_time - e.start_time, 2)
    return (
        f"[{e.event_type.upper()}] From t={e.start_time:.2f}s to t={e.end_time:.2f}s ({dur}s): "
        f"{e.explanation_seed}. "
        f"Peak magnitude: {e.max_magnitude:.3f}, mean magnitude: {e.mean_magnitude:.3f}."
    )


summaries = [summarize_event(e) for e in events]
print(f'Generated {len(summaries)} event summaries:\n')
for s in summaries:
    print(' •', s)

# ── Visualization ─────────────────────────────────────────────────────────────
EVENT_COLORS = {
    'idle': '#6EC6E6', 'walking_like': '#82C882',
    'impact': '#E85858', 'shaking': '#F5A623', 'unknown': '#CCCCCC'
}
ts = labeled_df['timestamp'].values

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle('Bosch Accelerometer — Ollama Query Engine', fontsize=13, fontweight='bold')

for col, color, lbl in zip(ACCEL_COLS, ['#4A90D9','#E87040','#5CB85C'], ['X','Y','Z']):
    axes[0].plot(ts, labeled_df[col], lw=0.7, alpha=0.8, color=color, label=f'Accel {lbl}')
axes[0].set_ylabel('Normalized')
axes[0].set_title('Accelerometer axes (z-scored)')
axes[0].legend(loc='upper right', fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].plot(ts, labeled_df['accel_magnitude'], lw=0.8, color='#333333', label='Magnitude', zorder=3)
seen = {}
for e in events:
    axes[1].axvspan(e.start_time, e.end_time, alpha=0.25,
                    color=EVENT_COLORS.get(e.event_type, '#CCCCCC'), label=e.event_type)
    seen.setdefault(e.event_type, True)
axes[1].set_ylabel('Magnitude')
axes[1].set_title('Magnitude + detected event bands')
handles, labels_ = axes[1].get_legend_handles_labels()
dedup = dict(zip(labels_, handles))
axes[1].legend(dedup.values(), dedup.keys(), loc='upper right', fontsize=8)
axes[1].grid(True, alpha=0.3)

axes[2].plot(ts, labeled_df['rolling_mean'], lw=1.0, color='#4A90D9', label='Rolling mean')
axes[2].plot(ts, labeled_df['rolling_std'],  lw=1.0, color='#E87040', label='Rolling std', linestyle='--')
axes[2].set_xlabel('Time (s)')
axes[2].set_ylabel('Value')
axes[2].set_title(f'Rolling statistics (window={ROLLING_WINDOW} samples)')
axes[2].legend(loc='upper right', fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('accelerometer_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

---
## 7. Configure LlamaIndex Global Settings

This is the key section that replaces the original OpenAI-based setup.

- `Settings.llm` → points at Ollama running locally on `localhost:11434`
- `Settings.embed_model` → uses `BAAI/bge-small-en-v1.5`, a fast 33M-parameter embedding model, downloaded once from HuggingFace Hub

Both run entirely on your Colab instance with no external API calls.

> **Swap the model:** Change `MODEL_NAME` to `'llama3.2:3b'` for higher answer quality (requires re-running §1 with the new model name).

In [ ]:
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# ── LLM: Ollama serving llama3.2:1b locally ───────────────────────────────────
# request_timeout: seconds to wait for a response — increase if the model is slow
Settings.llm = Ollama(
    model=MODEL_NAME,
    request_timeout=180.0,
    base_url='http://localhost:11434',   # default Ollama port
)

# ── Embeddings: HuggingFace bge-small-en-v1.5 (33M params, fast + accurate) ──
# Downloads ~130 MB on first run, cached afterwards in /root/.cache/huggingface
Settings.embed_model = HuggingFaceEmbedding(
    model_name='BAAI/bge-small-en-v1.5',
    embed_batch_size=32,
)

# Smoke test — generates a quick embedding to confirm the model loaded correctly
test_embed = Settings.embed_model.get_text_embedding('test')
print(f'✅ Embedding model ready — vector size: {len(test_embed)}')

# Smoke test — sends one token to Ollama
test_resp = Settings.llm.complete('Say OK')
print(f'✅ Ollama LLM ready — test response: "{str(test_resp).strip()}"')

---
## 8. Build the LlamaIndex VectorStoreIndex

Each event summary string becomes a `Document`. LlamaIndex embeds all documents using `bge-small-en-v1.5` and stores them in an in-memory vector index.

At query time, the query is embedded and the top-k most similar documents are retrieved, then sent to Ollama to synthesize a final answer.

In [ ]:
from llama_index.core import VectorStoreIndex, Document

# Wrap each summary string as a LlamaIndex Document
# Adding metadata makes it easier to filter by event type if needed later
documents = [
    Document(
        text=summary,
        metadata={
            'event_type':  events[i].event_type,
            'start_time':  events[i].start_time,
            'end_time':    events[i].end_time,
            'max_magnitude': events[i].max_magnitude,
        }
    )
    for i, summary in enumerate(summaries)
]

print(f'Indexing {len(documents)} event documents...')
index = VectorStoreIndex.from_documents(documents)

# similarity_top_k: how many documents to retrieve before passing to the LLM
# Increase to 5-8 if you have many events and want broader context
query_engine = index.as_query_engine(
    similarity_top_k=min(len(documents), 4),
    response_mode='compact',   # compact: condenses retrieved text before LLM call
)

print('✅ VectorStoreIndex built.')
print(f'   Documents indexed : {len(documents)}')
print(f'   LLM               : {MODEL_NAME} via Ollama')
print(f'   Embeddings        : BAAI/bge-small-en-v1.5')
print(f'   Retrieval top-k   : {min(len(documents), 4)}')

---
## 9. Query the Engine

The `query_events()` function sends a natural-language question to the full RAG pipeline:
1. Embed the question
2. Retrieve the top-k most relevant event summaries
3. Send retrieved context + question to `llama3.2:1b` via Ollama
4. Return the synthesized answer

Each query takes 5–30 seconds on a free T4 GPU depending on answer length.

In [ ]:
def query_events(question: str, verbose: bool = False) -> str:
    """
    Query the LlamaIndex engine and return the LLM's synthesized answer.
    Set verbose=True to also print the source documents that were retrieved.
    """
    response = query_engine.query(question)
    if verbose and hasattr(response, 'source_nodes'):
        print('  [Retrieved source documents:]')
        for node in response.source_nodes:
            print(f'    score={node.score:.3f} | {node.text[:100]}...')
        print()
    return str(response)


print('query_events() is ready. Run the next cell to test it.')

---
## 10. Example Queries

Five example queries covering the main use cases. Each prints the LLM's synthesized answer.

Expect real paragraph-style answers — not keyword-matched bullet points.

In [ ]:
EXAMPLE_QUERIES = [
    'What abnormal events happened and when did they occur?',
    'When was the device idle and for how long?',
    'Did any impact occur? Describe its severity.',
    'Was there any walking-like motion detected?',
    'Give me a complete summary of all detected motion events in chronological order.',
]

for q in EXAMPLE_QUERIES:
    print(f'\n❓ Query: "{q}"')
    print('─' * 70)
    answer = query_events(q)
    print(answer)

---
## 11. Interactive Query Cell

Type your own question here and run the cell to get an answer from the LLM.

In [ ]:
# ── Ask your own question ─────────────────────────────────────────────────────
# Change the string below and re-run this cell
MY_QUESTION = 'Which event had the highest peak acceleration?'

print(f'❓ {MY_QUESTION}')
print('─' * 70)
print(query_events(MY_QUESTION, verbose=True))

---
## 12. Save Outputs

Saves:
- `synthetic_accel_data.csv` — raw accelerometer readings
- `detected_events.csv` — events table with summaries
- `accelerometer_overview.png` — already saved in §6

To download files: click the **folder icon** in the left sidebar, right-click a file, select **Download**.

In [ ]:
# Save raw synthetic data
raw_df[['timestamp', 'accel_x', 'accel_y', 'accel_z', '_segment_label']].to_csv(
    'synthetic_accel_data.csv', index=False
)
print('✅ Saved synthetic_accel_data.csv')

# Save detected events with summaries
events_df = pd.DataFrame([{
    'event_type':       e.event_type,
    'start_time':       e.start_time,
    'end_time':         e.end_time,
    'duration_s':       round(e.end_time - e.start_time, 3),
    'max_magnitude':    e.max_magnitude,
    'mean_magnitude':   e.mean_magnitude,
    'explanation_seed': e.explanation_seed,
    'summary':          summarize_event(e),
} for e in events])
events_df.to_csv('detected_events.csv', index=False)
print('✅ Saved detected_events.csv')

print('\n--- Detected Events Table ---')
events_df[['event_type', 'start_time', 'end_time', 'duration_s', 'max_magnitude', 'mean_magnitude']]

---
## 13. Troubleshooting

| Symptom | Fix |
|---|---|
| `Connection refused` on Ollama | Re-run §1 — the server may have stopped. Check `/tmp/ollama.log` for errors. |
| Query takes >2 min | Switch to GPU: **Runtime > Change runtime type > T4 GPU** |
| `Model not found` error | Re-run the `ollama pull` line in §1 |
| Empty or very short answers | Increase `similarity_top_k` in §8, or switch to `llama3.2:3b` |
| HuggingFace download fails | Check internet connection; the model downloads once and is cached |
| `OutOfMemoryError` | Use `llama3.2:1b` (not 3b), or restart the runtime and re-run from §1 |

**Check Ollama server logs:**
```python
!tail -30 /tmp/ollama.log
```

**List available models:**
```python
!ollama list
```

**Test Ollama directly:**
```python
!ollama run llama3.2:1b "What is 2+2?"
```